In [1]:
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
import torchvision
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torchsummary import summary
from sklearn.metrics import confusion_matrix
import seaborn as sns
import numpy as np

/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/io/image.py:14: UserWarning: Failed to load image Python extension: 'dlopen(/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/image.so, 0x0006): Library not loaded: @rpath/libjpeg.9.dylib
  Referenced from: <EB3FF92A-5EB1-3EE8-AF8B-5923C1265422> /opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/image.so
  Reason: tried: '/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/site-packages/torchvision/../../../libjpeg.9.dylib' (no such file), '/opt/homebrew/Caskroom/miniconda/base/envs/dl/lib/python3.11/lib-dynload/../../libjpeg.9.dylib' (no such file), '/opt/homebrew/Caskroom/miniconda/base/envs/dl/bin/../lib/libjpeg.9.dylib' (no such file)'If you don't plan on using image functionality from `t

In [3]:
class PatchEmbedding(nn.Module):
    def __init__(self, img_size ,patch_size ,num_hiddens):
        super().__init__()
        self.num_patches= (img_size// patch_size ) ** 2
        self.conv = nn.Conv2d(num_hiddens,kernel_size=patch_size,stride=patch_size)

    def forward(self,x):
        return self.conv(x).flatten(2).transpose(1,2)
    

class ViTMLP(nn.Module):
    def __init__(self,mlp_num_hidden, mlp_num_outputs,dropout=0.5):
        super().__init__()
        self.dense1=nn.LazyLinear(mlp_num_hidden)
        self.gelu = nn.GELU()
        self.dropout1=nn.Dropout(dropout)
        self.dense2=nn.LazyLinear(mlp_num_outputs)
        self.dropout2=nn.Dropout(dropout)

    def forward(self,x):
        return self.dropout2(self.dense2(self.dropout1(self.gelu(self.dense1(x)))))
    

class ViTBlock(nn.Module):
    def __init__(self,num_hidden,norm_shape,mlp_num_hiddens,num_heads,dropout,use_bias=False):
        super().__init__()
        self.ln1= nn.LayerNorm(norm_shape)
        self.attention = nn.MultiheadAttention(num_hidden,num_heads,dropout,use_bias)
        self.ln2= nn.LayerNorm(norm_shape)
        self.mlp= ViTMLP(mlp_num_hiddens,num_hidden,dropout)

    def forward(self,x):
        x2= self.ln1(x)
        attention_output,_=self.attention(x,x,x)
        x = x + attention_output
        x2= self.ln2(x)
        mlp_output= self.mlp(x2)
        x = x + mlp_output
        return x 
    
